# Attention Mechanism

## Learning Objectives
1. Derive scaled dot-product attention from first principles using NumPy.
2. Implement multi-head attention in PyTorch with correct masking and projections.
3. Apply self-attention to a sentiment classification task and visualise attention heatmaps.
4. Compare cross-attention (encoder-decoder) versus self-attention patterns.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: Scaled Dot-Product Attention (NumPy)

In [ ]:
# Scaled dot-product attention: Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) V

def softmax_np(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    # Args:
    # Q: (seq_q, d_k)
    # K: (seq_k, d_k)
    # V: (seq_k, d_v)
    # mask: (seq_q, seq_k) boolean -- True positions are masked to -inf
    # Returns:
    # output: (seq_q, d_v), weights: (seq_q, seq_k)
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)         # (seq_q, seq_k)
    if mask is not None:
        scores[mask] = -1e9
    weights = softmax_np(scores, axis=-1)    # (seq_q, seq_k)
    output = weights @ V                     # (seq_q, d_v)
    return output, weights

# Toy example: 4 tokens, d_k=d_v=8
SEQ, D_K = 4, 8
rng = np.random.default_rng(0)
Q = rng.standard_normal((SEQ, D_K))
K = rng.standard_normal((SEQ, D_K))
V = rng.standard_normal((SEQ, D_K))

out, w = scaled_dot_product_attention(Q, K, V)
print(f"Output shape: {out.shape}")
print(f"Attention weights row sums (should all be 1.0): {w.sum(axis=-1).round(6)}")

# Causal (autoregressive) mask: token i cannot attend to future token j > i
causal_mask = np.triu(np.ones((SEQ, SEQ), dtype=bool), k=1)
out_causal, w_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print(f"Causal weights upper-triangle (should be ~0): {w_causal[0, 1:].round(6)}")
print("Scaled dot-product attention verified.")


## Level 2: Multi-Head Attention (PyTorch)

In [ ]:
class MultiHeadAttention(nn.Module):
    # Multi-head attention with optional causal masking.

    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        B, S, D = x.shape
        return x.view(B, S, self.n_heads, self.d_k).transpose(1, 2)   # (B,H,S,d_k)

    def forward(self, query, key, value, causal=False):
        B, S_q, _ = query.shape
        S_k = key.shape[1]
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))
        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)   # (B,H,S_q,S_k)
        if causal:
            mask = torch.triu(torch.ones(S_q, S_k, device=query.device), diagonal=1).bool()
            scores = scores.masked_fill(mask, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        context = weights @ V                                    # (B,H,S_q,d_k)
        context = context.transpose(1, 2).reshape(B, S_q, self.d_model)
        return self.W_o(context), weights

B, SEQ, D_MODEL, N_HEADS = 2, 10, 64, 4
x = torch.randn(B, SEQ, D_MODEL, device=device)
mha = MultiHeadAttention(D_MODEL, N_HEADS).to(device)

try:
    out, w = mha(x, x, x, causal=True)
except RuntimeError as exc:
    if "out of memory" in str(exc).lower():
        print("OOM -- reduce batch size or d_model")
        torch.cuda.empty_cache()
    raise

print(f"MHA output shape: {tuple(out.shape)}")
print(f"Attention weights shape: {tuple(w.shape)}  (B, heads, seq_q, seq_k)")
print(f"Causal check -- w[0,0,0,1:] should be 0: {w[0,0,0,1:].detach().cpu().numpy().round(4)}")


## Real-World Example 1: Self-Attention Sentiment + Heatmap

In [ ]:
# Simple self-attention classifier for toy sentiment; inspect attention weights.

class SentimentAttnModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_classes):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_bias = nn.Parameter(torch.zeros(1, 50, d_model))
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.clf = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, n_classes))

    def forward(self, ids):
        x = self.embed(ids) + self.pos_bias[:, :ids.size(1)]
        ctx, weights = self.attn(x, x, x)
        pooled = ctx.mean(dim=1)      # mean-pool over sequence
        return self.clf(pooled), weights

VOCAB, D, HEADS, SEQ_LEN = 100, 32, 4, 12
model_sent = SentimentAttnModel(VOCAB, D, HEADS, 2).to(device)

# Synthetic: class 1 if first token >= VOCAB//2
tokens = torch.randint(0, VOCAB, (200, SEQ_LEN))
labels = (tokens[:, 0] >= VOCAB // 2).long()
tr_ds = TensorDataset(tokens[:160].to(device), labels[:160].to(device))
tr_ld = DataLoader(tr_ds, batch_size=32, shuffle=True)
opt_sent = torch.optim.Adam(model_sent.parameters(), lr=1e-3)
crit_sent = nn.CrossEntropyLoss()

for epoch in range(30):
    model_sent.train()
    for xb, yb in tr_ld:
        opt_sent.zero_grad()
        logits, _ = model_sent(xb)
        crit_sent(logits, yb).backward()
        opt_sent.step()

model_sent.eval()
with torch.no_grad():
    logits, attn_w = model_sent(tokens[160:161].to(device))
    pred = logits.argmax(dim=-1).item()
print(f"Predicted class: {pred}  | True class: {labels[160].item()}")
print(f"Attention heatmap (head 0) shape: {tuple(attn_w[0,0].shape)}")
row_max = attn_w[0, 0].cpu().numpy().max(axis=-1)
print(f"Max attention per query position: {row_max.round(3)}")
print("High weight on position 0 confirms model uses the class-discriminating token.")


## Real-World Example 2: Cross-Attention Encoder-Decoder

In [ ]:
# Cross-attention: decoder queries attend over encoder key-values.
# Fundamental to seq2seq models (translation, summarisation, captioning).

class EncoderDecoder(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ff   = nn.Sequential(nn.Linear(d_model, d_model*4), nn.GELU(),
                                   nn.Linear(d_model*4, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def encode(self, src):
        ctx, _ = self.self_attn(src, src, src)
        return self.norm1(src + ctx)

    def decode(self, tgt, memory):
        # Causal self-attention on target (decoder) sequence
        ctx1, _ = self.self_attn(tgt, tgt, tgt, causal=True)
        tgt2 = self.norm1(tgt + ctx1)
        # Cross-attention: decoder queries attend over encoder memory
        ctx2, cross_w = self.cross_attn(tgt2, memory, memory)
        tgt3 = self.norm2(tgt2 + ctx2)
        return self.norm3(tgt3 + self.ff(tgt3)), cross_w

ENC_SEQ, DEC_SEQ, D, H = 15, 10, 64, 4
enc_dec = EncoderDecoder(D, H).to(device)
src = torch.randn(2, ENC_SEQ, D, device=device)
tgt = torch.randn(2, DEC_SEQ, D, device=device)

memory = enc_dec.encode(src)
output, cross_weights = enc_dec.decode(tgt, memory)
print(f"Encoder memory:  {tuple(memory.shape)}")
print(f"Decoder output:  {tuple(output.shape)}")
print(f"Cross-attn weights: {tuple(cross_weights.shape)}  (B, heads, dec_seq, enc_seq)")
avg_cross = cross_weights.mean(dim=(0,1)).detach().cpu().numpy()
print(f"Peak encoder position per decoder step: {avg_cross.argmax(axis=-1)}")


## Real-World Example 3: Attention Pattern Visualization

In [ ]:
# Visualise how attention patterns differ: uniform vs peaked vs causal.
# Useful for debugging transformer behaviour in production.

def make_attention_pattern(kind, seq_len=8, d_k=16):
    rng2 = np.random.default_rng(99)
    if kind == "uniform":
        # Very small Q.K products -> nearly uniform softmax
        Q = rng2.standard_normal((seq_len, d_k)) * 0.01
        K = rng2.standard_normal((seq_len, d_k)) * 0.01
        mask = None
    elif kind == "peaked":
        # Aligned Q=K: each token strongly attends to itself
        Q = np.eye(seq_len, d_k)
        K = np.eye(seq_len, d_k)
        mask = None
    elif kind == "causal":
        Q = rng2.standard_normal((seq_len, d_k))
        K = rng2.standard_normal((seq_len, d_k))
        mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
    else:
        raise ValueError(kind)
    _, w = scaled_dot_product_attention(Q, K, np.eye(seq_len), mask=mask)
    return w

for kind in ("uniform", "peaked", "causal"):
    pat = make_attention_pattern(kind)
    diag_sum = np.diag(pat).sum()
    entropy = -(pat * np.log(pat + 1e-9)).sum(axis=-1).mean()
    print(f"{kind:8s}  diagonal_sum={diag_sum:.3f}  avg_entropy={entropy:.3f}")

print()
print("Pattern interpretation:")
print("  Uniform  -- model has not learned to focus (high entropy)")
print("  Peaked   -- strong local self-attention (first layers)")
print("  Causal   -- autoregressive; later tokens use richer context (lower entropy)")
